# Tree Models and Validation Model Selection — src Refactor

This notebook compares Logistic Regression, Random Forest, and XGBoost, performs patient-grouped cross-validation and lightweight tuning, and selects the candidate model **using validation data only**.

Reusable model builders and evaluation metrics are imported from `src/`.

## 1. Bootstrap project imports

In [1]:
from pathlib import Path
import sys

# Make imports work whether VS Code starts the notebook from project root
# or from the notebooks/ directory.
cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_DIR = cwd
elif (cwd.parent / "src").exists():
    PROJECT_DIR = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing src/. "
        "Open this notebook from the clinical-outcome-prediction project."
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project root:", PROJECT_DIR)
print("Python:", sys.executable)

Project root: C:\Projects\clinical-outcome-prediction
Python: c:\Projects\clinical-outcome-prediction\.venv\Scripts\python.exe


## 2. Imports

In [2]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedGroupKFold,
    cross_validate,
)

from src.config import (
    MODELS_DIR,
    TABLES_DIR,
    PREDICTIONS_DIR,
    PREPROCESSOR_PATH,
    FEATURE_CONFIG_PATH,
    SELECTED_MODEL_PATH,
    SELECTED_MODEL_CONFIG_PATH,
)
from src.data import (
    load_modeling_splits,
    load_feature_config,
    validate_patient_split,
)
from src.models import (
    build_random_forest,
    build_xgboost,
    calculate_scale_pos_weight,
)
from src.evaluation import classification_metrics

RANDOM_STATE = 42
DEFAULT_THRESHOLD = 0.50

LOGISTIC_MODEL_PATH = MODELS_DIR / "logistic_regression.joblib"
RF_MODEL_PATH = MODELS_DIR / "random_forest.joblib"
XGB_MODEL_PATH = MODELS_DIR / "xgboost.joblib"
TUNED_RF_MODEL_PATH = MODELS_DIR / "random_forest_tuned.joblib"
TUNED_XGB_MODEL_PATH = MODELS_DIR / "xgboost_tuned.joblib"

for directory in [
    MODELS_DIR,
    TABLES_DIR,
    PREDICTIONS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

## 3. Load data and saved preprocessing

In [3]:
train_df, validation_df, test_df = load_modeling_splits()
validate_patient_split(train_df, validation_df, test_df)

feature_config = load_feature_config()
preprocessor = joblib.load(PREPROCESSOR_PATH)
logistic_model = joblib.load(LOGISTIC_MODEL_PATH)

target_column = feature_config["target_column"]
feature_columns = feature_config["retained_feature_columns"]

X_train = train_df[feature_columns].copy()
X_validation = validation_df[feature_columns].copy()

y_train = train_df[target_column].copy()
y_validation = validation_df[target_column].copy()

training_groups = train_df["subject_id"].copy()

scale_pos_weight = calculate_scale_pos_weight(
    y_train
)

print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("scale_pos_weight:", scale_pos_weight)

Train: (89, 66)
Validation: (20, 66)
scale_pos_weight: 7.090909090909091


## 4. Build and train untuned tree models

In [4]:
random_forest_model = build_random_forest(
    preprocessor,
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=5,
    class_weight="balanced",
)

xgb_model = build_xgboost(
    preprocessor,
    scale_pos_weight=scale_pos_weight,
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
)

random_forest_model.fit(
    X_train,
    y_train,
)

xgb_model.fit(
    X_train,
    y_train,
)

print("Random Forest and XGBoost training completed.")

Random Forest and XGBoost training completed.


## 5. Generate untuned validation probabilities

In [5]:
logistic_validation_probability = (
    logistic_model.predict_proba(X_validation)[:, 1]
)

rf_validation_probability = (
    random_forest_model.predict_proba(X_validation)[:, 1]
)

xgb_validation_probability = (
    xgb_model.predict_proba(X_validation)[:, 1]
)

untuned_validation_results = pd.DataFrame(
    [
        {
            "model": "Logistic Regression",
            **classification_metrics(
                y_validation,
                logistic_validation_probability,
                threshold=DEFAULT_THRESHOLD,
            ),
        },
        {
            "model": "Random Forest",
            **classification_metrics(
                y_validation,
                rf_validation_probability,
                threshold=DEFAULT_THRESHOLD,
            ),
        },
        {
            "model": "XGBoost",
            **classification_metrics(
                y_validation,
                xgb_validation_probability,
                threshold=DEFAULT_THRESHOLD,
            ),
        },
    ]
).sort_values(
    ["auprc", "auroc"],
    ascending=False,
).reset_index(drop=True)

untuned_validation_results

,model,auroc,auprc,threshold,accuracy,precision_ppv,recall_sensitivity,specificity,negative_predictive_value,f1_score,true_negatives,false_positives,false_negatives,true_positives,patients_flagged,flagged_percentage
0,Random Forest,0.916667,0.700000,0.5,0.80,0.333333,1.0,0.777778,1.000000,0.5,14,4,0,2,6,0.30
1,XGBoost,0.722222,0.266667,0.5,0.80,0.000000,0.0,0.888889,0.888889,0.0,16,2,2,0,2,0.10
2,Logistic Regression,0.361111,0.133333,0.5,0.85,0.000000,0.0,0.944444,0.894737,0.0,17,1,2,0,1,0.05


## 6. Patient-grouped cross-validation

In [6]:
minimum_class_count = int(
    y_train.value_counts().min()
)

n_splits = min(
    5,
    minimum_class_count,
)

if n_splits < 2:
    raise ValueError(
        "Not enough observations in both classes for grouped CV."
    )

grouped_cv = StratifiedGroupKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scoring = {
    "auroc": "roc_auc",
    "auprc": "average_precision",
}

rf_cv_scores = cross_validate(
    clone(random_forest_model),
    X_train,
    y_train,
    groups=training_groups,
    cv=grouped_cv,
    scoring=scoring,
    n_jobs=-1,
)

xgb_cv_scores = cross_validate(
    clone(xgb_model),
    X_train,
    y_train,
    groups=training_groups,
    cv=grouped_cv,
    scoring=scoring,
    n_jobs=-1,
)

cv_summary = pd.DataFrame(
    [
        {
            "model": "Random Forest",
            "mean_cv_auroc": np.mean(rf_cv_scores["test_auroc"]),
            "sd_cv_auroc": np.std(rf_cv_scores["test_auroc"], ddof=1),
            "mean_cv_auprc": np.mean(rf_cv_scores["test_auprc"]),
            "sd_cv_auprc": np.std(rf_cv_scores["test_auprc"], ddof=1),
        },
        {
            "model": "XGBoost",
            "mean_cv_auroc": np.mean(xgb_cv_scores["test_auroc"]),
            "sd_cv_auroc": np.std(xgb_cv_scores["test_auroc"], ddof=1),
            "mean_cv_auprc": np.mean(xgb_cv_scores["test_auprc"]),
            "sd_cv_auprc": np.std(xgb_cv_scores["test_auprc"], ddof=1),
        },
    ]
)

cv_summary

,model,mean_cv_auroc,sd_cv_auroc,mean_cv_auprc,sd_cv_auprc
0,Random Forest,0.689583,0.159385,0.494722,0.117130
1,XGBoost,0.719167,0.183211,0.472241,0.168811


## 7. Lightweight Random Forest tuning

In [7]:
rf_parameter_distributions = {
    "classifier__n_estimators": [300, 500, 800],
    "classifier__max_depth": [5, 8, 12, None],
    "classifier__min_samples_leaf": [2, 5, 10],
    "classifier__max_features": ["sqrt", "log2", 0.7],
}

rf_search = RandomizedSearchCV(
    estimator=build_random_forest(
        preprocessor,
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=5,
    ),
    param_distributions=rf_parameter_distributions,
    n_iter=12,
    scoring="average_precision",
    cv=grouped_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
)

rf_search.fit(
    X_train,
    y_train,
    groups=training_groups,
)

tuned_rf_model = rf_search.best_estimator_

print("Best RF CV AUPRC:", rf_search.best_score_)
print("Best RF parameters:", rf_search.best_params_)

Best RF CV AUPRC: 0.5117647058823529
Best RF parameters: {'classifier__n_estimators': 300, 'classifier__min_samples_leaf': 5, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 8}


## 8. Lightweight XGBoost tuning

In [8]:
xgb_parameter_distributions = {
    "classifier__n_estimators": [300, 500, 800],
    "classifier__learning_rate": [0.02, 0.03, 0.05, 0.08],
    "classifier__max_depth": [2, 3, 4, 5],
    "classifier__min_child_weight": [1, 3, 5, 8],
    "classifier__subsample": [0.7, 0.8, 1.0],
    "classifier__colsample_bytree": [0.7, 0.8, 1.0],
}

xgb_search = RandomizedSearchCV(
    estimator=build_xgboost(
        preprocessor,
        scale_pos_weight=scale_pos_weight,
    ),
    param_distributions=xgb_parameter_distributions,
    n_iter=16,
    scoring="average_precision",
    cv=grouped_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
)

xgb_search.fit(
    X_train,
    y_train,
    groups=training_groups,
)

tuned_xgb_model = xgb_search.best_estimator_

print("Best XGB CV AUPRC:", xgb_search.best_score_)
print("Best XGB parameters:", xgb_search.best_params_)

Best XGB CV AUPRC: 0.550375816993464
Best XGB parameters: {'classifier__subsample': 0.7, 'classifier__n_estimators': 800, 'classifier__min_child_weight': 3, 'classifier__max_depth': 4, 'classifier__learning_rate': 0.02, 'classifier__colsample_bytree': 0.7}


## 9. Compare tuned and untuned candidates on validation data

In [9]:
tuned_rf_validation_probability = (
    tuned_rf_model.predict_proba(
        X_validation
    )[:, 1]
)

tuned_xgb_validation_probability = (
    tuned_xgb_model.predict_proba(
        X_validation
    )[:, 1]
)

candidate_models = {
    "Logistic Regression": (
        logistic_model,
        logistic_validation_probability,
    ),
    "Random Forest": (
        random_forest_model,
        rf_validation_probability,
    ),
    "XGBoost": (
        xgb_model,
        xgb_validation_probability,
    ),
    "Tuned Random Forest": (
        tuned_rf_model,
        tuned_rf_validation_probability,
    ),
    "Tuned XGBoost": (
        tuned_xgb_model,
        tuned_xgb_validation_probability,
    ),
}

complete_validation_results = pd.DataFrame(
    [
        {
            "model": model_name,
            **classification_metrics(
                y_validation,
                probability,
                threshold=DEFAULT_THRESHOLD,
            ),
        }
        for model_name, (_, probability)
        in candidate_models.items()
    ]
).sort_values(
    ["auprc", "auroc"],
    ascending=False,
).reset_index(drop=True)

complete_validation_results

,model,auroc,auprc,threshold,accuracy,precision_ppv,recall_sensitivity,specificity,negative_predictive_value,f1_score,true_negatives,false_positives,false_negatives,true_positives,patients_flagged,flagged_percentage
0,Tuned Random Forest,0.944444,0.750000,0.5,0.80,0.333333,1.0,0.777778,1.000000,0.5,14,4,0,2,6,0.30
1,Random Forest,0.916667,0.700000,0.5,0.80,0.333333,1.0,0.777778,1.000000,0.5,14,4,0,2,6,0.30
2,XGBoost,0.722222,0.266667,0.5,0.80,0.000000,0.0,0.888889,0.888889,0.0,16,2,2,0,2,0.10
3,Tuned XGBoost,0.611111,0.183333,0.5,0.80,0.000000,0.0,0.888889,0.888889,0.0,16,2,2,0,2,0.10
4,Logistic Regression,0.361111,0.133333,0.5,0.85,0.000000,0.0,0.944444,0.894737,0.0,17,1,2,0,1,0.05


## 10. Select model using validation AUPRC, then AUROC

In [10]:
top_row = complete_validation_results.iloc[0]

selected_model_name = str(
    top_row["model"]
)

selected_model, selected_validation_probability = (
    candidate_models[
        selected_model_name
    ]
)

print("Selected candidate model:", selected_model_name)
print("Validation AUPRC:", top_row["auprc"])
print("Validation AUROC:", top_row["auroc"])

Selected candidate model: Tuned Random Forest
Validation AUPRC: 0.75
Validation AUROC: 0.9444444444444444


## 11. Save all models and selection metadata

In [11]:
joblib.dump(random_forest_model, RF_MODEL_PATH)
joblib.dump(xgb_model, XGB_MODEL_PATH)
joblib.dump(tuned_rf_model, TUNED_RF_MODEL_PATH)
joblib.dump(tuned_xgb_model, TUNED_XGB_MODEL_PATH)
joblib.dump(selected_model, SELECTED_MODEL_PATH)

selection_config = {
    "selected_model_name": selected_model_name,
    "selection_dataset": "validation",
    "primary_selection_metric": "AUPRC",
    "secondary_selection_metric": "AUROC",
    "validation_auprc": float(top_row["auprc"]),
    "validation_auroc": float(top_row["auroc"]),
    "test_set_used_for_selection": False,
}

with open(
    SELECTED_MODEL_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        selection_config,
        file,
        indent=2,
    )

complete_validation_results.to_csv(
    TABLES_DIR / "validation_complete_model_comparison.csv",
    index=False,
)

cv_summary.to_csv(
    TABLES_DIR / "untuned_tree_cv_summary.csv",
    index=False,
)

validation_predictions = validation_df[
    ["subject_id", "hadm_id", "stay_id", target_column]
].copy()

for model_name, (_, probability) in candidate_models.items():
    safe_name = (
        model_name.lower()
        .replace(" ", "_")
    )
    validation_predictions[
        f"{safe_name}_probability"
    ] = probability

validation_predictions.to_csv(
    PREDICTIONS_DIR / "validation_tree_model_predictions.csv",
    index=False,
)

print("Model artifacts and validation results saved.")

Model artifacts and validation results saved.


## 12. Reload selected model

In [12]:
loaded_selected_model = joblib.load(
    SELECTED_MODEL_PATH
)

reloaded_probability = (
    loaded_selected_model.predict_proba(
        X_validation
    )[:, 1]
)

assert np.allclose(
    reloaded_probability,
    selected_validation_probability,
)

print("Selected model reload validation passed.")

Selected model reload validation passed.


## Notebook 7 summary

Model selection used the validation set only. The untouched test set remains reserved for Notebook 8.

Next: **Notebook 8 — Probability calibration and final test evaluation**.